# Azure ETL Pipeline — Medallion Architecture
**Bronze → Silver → Gold using PySpark + Delta Lake**

| Layer | Description |
|-------|-------------|
| Bronze | Raw data as ingested from source |
| Silver | Cleaned, deduplicated, type-cast data |
| Gold   | Aggregated, joined, business-ready data |

## 🔧 CELL 1 — Configuration: Set your storage account details here

In [ ]:
# ============================================================
# CONFIGURATION — Update these values for your Azure account
# ============================================================

STORAGE_ACCOUNT = "<your_storage_account_name>"   # e.g. myetlstorage
CONTAINER       = "medallion"                      # your ADLS container name
SAS_TOKEN       = "<your_sas_token>"               # from Azure Portal > Storage > SAS

# Set Spark config to authenticate with ADLS using SAS token
spark.conf.set(
    f"fs.azure.sas.{CONTAINER}.{STORAGE_ACCOUNT}.blob.core.windows.net",
    SAS_TOKEN
)

# Base path shortcut
BASE = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

print(f"✅ Config set. Base path: {BASE}")

## 🥉 CELL 2 — BRONZE LAYER: Read raw JSON files

In [ ]:
# ============================================================
# BRONZE LAYER — Read raw data as-is from ADLS bronze folder
# ADF Copy Activity already moved files here from raw/
# ============================================================

df_customers    = spark.read.option("multiline", "true").json(f"{BASE}/bronze/customers.json")
df_products     = spark.read.option("multiline", "true").json(f"{BASE}/bronze/products.json")
df_transactions = spark.read.option("multiline", "true").json(f"{BASE}/bronze/transactions.json")
df_stores       = spark.read.option("multiline", "true").json(f"{BASE}/bronze/stores.json")

print("✅ Bronze layer loaded successfully")
print(f"   Customers    : {df_customers.count()} rows")
print(f"   Products     : {df_products.count()} rows")
print(f"   Transactions : {df_transactions.count()} rows")
print(f"   Stores       : {df_stores.count()} rows")

# Preview
print("\n--- Customers Schema ---")
df_customers.printSchema()
print("\n--- Transactions Sample ---")
df_transactions.show(5, truncate=False)

## 🥈 CELL 3 — SILVER LAYER: Clean, cast types, deduplicate

In [ ]:
from pyspark.sql.functions import col, to_date, when, upper, trim
from pyspark.sql.types import IntegerType, DoubleType

# ============================================================
# SILVER — CUSTOMERS
# Clean: fill nulls, trim strings, cast types
# ============================================================
df_silver_customers = df_customers \
    .fillna({"age": 0, "email": "unknown@email.com"}) \
    .withColumn("customer_id",  col("customer_id").cast(IntegerType())) \
    .withColumn("age",          col("age").cast(IntegerType())) \
    .withColumn("name",         trim(col("name"))) \
    .withColumn("membership",   upper(trim(col("membership")))) \
    .withColumn("signup_date",  to_date(col("signup_date"), "yyyy-MM-dd")) \
    .dropDuplicates(["customer_id"])

# ============================================================
# SILVER — PRODUCTS
# Clean: fill nulls, cast types
# ============================================================
df_silver_products = df_products \
    .fillna({"stock_quantity": 0, "rating": 0.0}) \
    .withColumn("product_id",     col("product_id").cast(IntegerType())) \
    .withColumn("price",          col("price").cast(DoubleType())) \
    .withColumn("stock_quantity", col("stock_quantity").cast(IntegerType())) \
    .withColumn("rating",         col("rating").cast(DoubleType())) \
    .withColumn("launch_date",    to_date(col("launch_date"), "yyyy-MM-dd")) \
    .dropDuplicates(["product_id"])

# ============================================================
# SILVER — TRANSACTIONS
# Clean: cast types, fill nulls
# ============================================================
df_silver_transactions = df_transactions \
    .fillna({"discount": 0.0}) \
    .withColumn("transaction_id",   col("transaction_id").cast(IntegerType())) \
    .withColumn("customer_id",      col("customer_id").cast(IntegerType())) \
    .withColumn("product_id",       col("product_id").cast(IntegerType())) \
    .withColumn("store_id",         col("store_id").cast(IntegerType())) \
    .withColumn("quantity",         col("quantity").cast(IntegerType())) \
    .withColumn("discount",         col("discount").cast(DoubleType())) \
    .withColumn("transaction_date", to_date(col("transaction_date"), "yyyy-MM-dd")) \
    .dropDuplicates(["transaction_id"])

# ============================================================
# SILVER — STORES
# ============================================================
df_silver_stores = df_stores \
    .withColumn("store_id",   col("store_id").cast(IntegerType())) \
    .withColumn("open_date",  to_date(col("open_date"), "yyyy-MM-dd")) \
    .withColumn("region",     upper(trim(col("region")))) \
    .dropDuplicates(["store_id"])

print("✅ Silver layer transformations complete")
df_silver_transactions.show(5, truncate=False)

## 🥈 CELL 4 — Write Silver layer to Delta Lake

In [ ]:
# Write all 4 cleaned dataframes to Silver as Delta format

df_silver_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{BASE}/silver/customers")

df_silver_products.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{BASE}/silver/products")

df_silver_transactions.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{BASE}/silver/transactions")

df_silver_stores.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{BASE}/silver/stores")

print("✅ Silver Delta tables written to ADLS")

## 🥇 CELL 5 — GOLD LAYER: Join + Aggregate business logic

In [ ]:
from pyspark.sql.functions import sum as _sum, avg, count, round as _round

# Read from Silver Delta tables
df_s_customers    = spark.read.format("delta").load(f"{BASE}/silver/customers")
df_s_products     = spark.read.format("delta").load(f"{BASE}/silver/products")
df_s_transactions = spark.read.format("delta").load(f"{BASE}/silver/transactions")
df_s_stores       = spark.read.format("delta").load(f"{BASE}/silver/stores")

# ============================================================
# GOLD TABLE 1 — df_gold (Retail Sales Summary)
# Join: transactions + products + customers + stores
# ============================================================
df_joined = df_s_transactions \
    .join(df_s_products,     on="product_id",  how="left") \
    .join(df_s_customers,    on="customer_id", how="left") \
    .join(df_s_stores,       on="store_id",    how="left")

# Add revenue column: (price * quantity) - discount
from pyspark.sql.functions import expr
df_joined = df_joined.withColumn(
    "revenue",
    _round((col("price") * col("quantity")) - col("discount"), 2)
)

# ============================================================
# GOLD TABLE 2 — Sales by Product Category
# ============================================================
df_gold_by_category = df_joined.groupBy("category").agg(
    count("transaction_id").alias("total_transactions"),
    _sum("quantity").alias("total_units_sold"),
    _round(_sum("revenue"), 2).alias("total_revenue"),
    _round(avg("revenue"), 2).alias("avg_revenue_per_sale")
).orderBy(col("total_revenue").desc())

# ============================================================
# GOLD TABLE 3 — Sales by Store Region
# ============================================================
df_gold_by_region = df_joined.groupBy("region", "store_name").agg(
    count("transaction_id").alias("total_transactions"),
    _round(_sum("revenue"), 2).alias("total_revenue")
).orderBy(col("total_revenue").desc())

# ============================================================
# GOLD TABLE 4 — Customer Lifetime Value by Membership Tier
# ============================================================
df_gold_clv = df_joined.groupBy("membership").agg(
    count("transaction_id").alias("total_orders"),
    _round(_sum("revenue"), 2).alias("total_revenue"),
    _round(avg("revenue"), 2).alias("avg_order_value")
).orderBy(col("total_revenue").desc())

print("✅ Gold layer transformations complete")
print("\n--- Sales by Category ---")
df_gold_by_category.show(truncate=False)
print("\n--- Sales by Region ---")
df_gold_by_region.show(truncate=False)
print("\n--- Customer Lifetime Value ---")
df_gold_clv.show(truncate=False)

## 🥇 CELL 6 — Write Gold layer to Delta Lake

In [ ]:
# Write Gold tables to Delta Lake — these are the final reporting tables

df_gold_by_category.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{BASE}/gold/sales_by_category")

df_gold_by_region.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{BASE}/gold/sales_by_region")

df_gold_clv.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{BASE}/gold/customer_lifetime_value")

print("✅ Gold Delta tables written to ADLS")
print(f"\n📁 Gold layer paths:")
print(f"   {BASE}/gold/sales_by_category")
print(f"   {BASE}/gold/sales_by_region")
print(f"   {BASE}/gold/customer_lifetime_value")

## ✅ Pipeline Complete!

| Layer | Tables Written |
|-------|----------------|
| 🥉 Bronze | customers, products, transactions, stores (raw JSON) |
| 🥈 Silver | customers, products, transactions, stores (cleaned Delta) |
| 🥇 Gold   | sales_by_category, sales_by_region, customer_lifetime_value |